# Importations

In [12]:
# Numerical and scientific python programming
import numpy as np

import matplotlib.pyplot as plt

# Auxiliary python functions
from functools import reduce
import time as t

# Definitions

In [24]:
def adj(A: np.ndarray) -> np.ndarray:
    return np.conj(A).T

def tensor_product(O_v: Sequence[np.ndarray]) -> np.ndarray:
    try:
        return reduce(np.kron, O_v)
    except ValueError as e:
        raise ValueError("Some arrays in O_v have incompatible shapes for Kronecker product.") from e

def compute_outer_product(psi: np.ndarray, phi: np.ndarray) -> np.ndarray:
    if psi.shape != phi.shape:
        raise ValueError("psi and phi must have the same shape.")
    if len(psi.shape) != 2 or psi.shape[1] != 1:
        raise ValueError("psi and phi must be column vectors of shape (d, 1).")
    return psi @ adj(phi)

def rand_IO(d: int) -> np.ndarray:
    rng = np.random.default_rng()
    while True:
        mat = rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))
        if np.linalg.matrix_rank(mat) == d:
            return mat

def rand_LIO(dims: Sequence[int]) -> np.ndarray:
    ops = [rand_IO(d) for d in dims]
    return tensor_product(ops)

def compute_GHZ_state(N: int) -> np.ndarray:
    if N <= 0:
        return np.zeros((2**max(0, N), 1), dtype=complex)
    GHZ = np.zeros((2**N, 1), dtype=complex)
    GHZ[0, 0] = 1
    GHZ[2**N - 1, 0] = 1
    return GHZ / np.sqrt(2)

def compute_W_state(N: int) -> np.ndarray:
    if N <= 0:
        return np.zeros((2**max(0, N), 1), dtype=complex)
    W = np.zeros((2**N, 1), dtype=complex)
    for n in range(N):
        W[2**n, 0] = 1
    return W / np.sqrt(N)

# Classes

In [25]:
Cl = ['Sep', 'Bi_12', 'Bi_13', 'Bi_23', 'W', 'GHZ']

C = len(Cl)

print('\nClasses:\n')

for c in Cl:

    print(c)

print('\nNumber of classes: ', C)


Classes:

Sep
Bi_12
Bi_13
Bi_23
W
GHZ

Number of classes:  6


# Random dataset generation

In [27]:
dn, N = 2, 3

dim = [dn] * N
d = int(np.prod(dim))

phi_0 = np.array([[1], [0]])
phi_1 = np.array([[0], [1]])

E_c = 100

t_1 = t.time()

RHO = {}

for c in Cl:
    
    rho = np.zeros((E_c, d, d), dtype = np.complex128)

    for e in range(E_c):
            
        if c == 'Sep':
            psi_e = tensor_product(N * [phi_0])
        
        elif c == 'Bi_12':
            psi_e = (1 / np.sqrt(2)) * (tensor_product(N * [phi_0]) + tensor_product([phi_1, phi_1, phi_0]))
        
        elif c == 'Bi_13':
            psi_e = (1 / np.sqrt(2)) * (tensor_product(N * [phi_0]) + tensor_product([phi_1, phi_0, phi_1]))
        
        elif c == 'Bi_23':
            psi_e = (1 / np.sqrt(2)) * (tensor_product(N * [phi_0]) + tensor_product([phi_0, phi_1, phi_1]))
        
        elif c == 'W':
            psi_e = compute_W_state(N)
        
        elif c == 'GHZ':
            psi_e = compute_GHZ_state(N)
    
        psi_e = rand_LIO(dim) @ psi_e
        psi_e = psi_e / np.linalg.norm(psi_e)
        
        rho[e] = compute_outer_product(psi_e, psi_e)
    
    RHO[c] = rho

t_2 = t.time()
print(f'Time to generate density matrices: {np.round(t_2 - t_1, 2)} s')

Time to generate density matrices: 0.08 s
